Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap
import importlib

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error

from xgboost import XGBRegressor

import dataset_create
importlib.reload(dataset_create)
from dataset_create import FEATURES, ORIGINES, charger, split

In [ ]:
from dataset_create import charger

df = charger()
df.head()

Features enrichies: calendar, events, and historical traffic lags

In [ ]:
features = FEATURES
features

Backtest temporel sur l'horizon de 120 heures

In [ ]:
BACKTEST_ORIGIN = ORIGINES[-3]  # latest origin with a complete observed horizon in this dataset

target_debit = "debit"
target_occupation = "occupation"

In [ ]:
categorical_features = ["axe", "vacances"]
numeric_features = [col for col in features if col not in categorical_features]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", "passthrough", numeric_features),
    ]
)

In [ ]:
print("Nombre de lignes :", len(df))
print("Period:", df["datetime"].min(), "->", df["datetime"].max())
print("Features:", features)
print("Colonnes externes disponibles :", [c for c in ["vacances", "ferie", "pont", "paris_respire", "evenement", "confinement"] if c in df])

In [ ]:
X_train_debit, X_test_debit, y_train_debit, y_test_debit = split(
    df,
    BACKTEST_ORIGIN,
    cible=target_debit,
    features=features,
)

print("Origine du backtest :", BACKTEST_ORIGIN)
print("Debit train:", X_train_debit.shape, "| Test d?bit :", X_test_debit.shape)

In [ ]:
print("NaN dans la cible train :", y_train_debit.isna().sum())
print("Inf dans la cible train :", np.isinf(y_train_debit).sum())
print(y_train_debit.describe())

XGBoost

In [ ]:
def create_model():

    xgb = XGBRegressor(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    )

    model = Pipeline([
        ("preprocessor", preprocessor),
        ("xgb", xgb)
    ])

    return model

In [ ]:
print("Categorical features:", categorical_features)
print("Numeric features:", numeric_features)
print("Train/test d?bit :", X_train_debit.shape, X_test_debit.shape)
print("Test axes:", X_test_debit["axe"].value_counts().to_dict())

In [ ]:
## Débit
model_debit = create_model()

model_debit.fit(X_train_debit, y_train_debit)

In [ ]:
X_train_occupation, X_test_occupation, y_train_occupation, y_test_occupation = split(
    df,
    BACKTEST_ORIGIN,
    cible=target_occupation,
    features=features,
)

print("Occupation train:", X_train_occupation.shape, "| Test occupation :", X_test_occupation.shape)

In [ ]:
## Occupation
model_occupation = create_model()

model_occupation.fit(X_train_occupation, y_train_occupation)

RMSE

In [ ]:
# ?valuation sur les 120 heures suivant l'origine du backtest
pred_debit = model_debit.predict(X_test_debit)
pred_occupation = model_occupation.predict(X_test_occupation)

rmse_debit = np.sqrt(mean_squared_error(y_test_debit, pred_debit))
rmse_occupation = np.sqrt(mean_squared_error(y_test_occupation, pred_occupation))

print(f"Debit RMSE ({BACKTEST_ORIGIN}) : {rmse_debit:.4f}")
print(f"Occupation RMSE ({BACKTEST_ORIGIN}) : {rmse_occupation:.4f}")

In [ ]:
# Baseline : pr?dire la moyenne de chaque jeu d'entra?nement
baseline_debit = np.full(len(y_test_debit), y_train_debit.mean())
baseline_occupation = np.full(len(y_test_occupation), y_train_occupation.mean())

rmse_baseline_debit = np.sqrt(mean_squared_error(y_test_debit, baseline_debit))
rmse_baseline_occupation = np.sqrt(mean_squared_error(y_test_occupation, baseline_occupation))

print("Comparaison avec la baseline")
print(f"D?bit       : XGBoost = {rmse_debit:.2f} | Baseline = {rmse_baseline_debit:.2f}")
print(f"Occupation  : XGBoost = {rmse_occupation:.2f} | Baseline = {rmse_baseline_occupation:.2f}")

SHAP

In [ ]:
def analyser_shap(model, X_test, nom_cible):

    # On prend un échantillon du jeu de test
    X_sample = X_test.sample(
        n=min(1000, len(X_test)),
        random_state=42
    )

    # Récupération du préprocesseur et du modèle XGBoost
    preprocessor = model.named_steps["preprocessor"]
    xgb_model = model.named_steps["xgb"]

    # Transformation des données
    X_transformed = preprocessor.transform(X_sample)

    # Conversion en tableau dense si nécessaire
    if hasattr(X_transformed, "toarray"):
        X_transformed = X_transformed.toarray()

    # Récupération des noms des features après encodage
    feature_names = preprocessor.get_feature_names_out()

    X_shap = pd.DataFrame(
        X_transformed,
        columns=feature_names
    )

    # Calcul des valeurs SHAP
    explainer = shap.TreeExplainer(xgb_model)
    shap_values = explainer(X_shap)

    # Importance globale
    plt.figure()
    shap.plots.bar(shap_values, max_display=15, show=False)
    plt.title(f"Importance des features - {nom_cible}")
    plt.tight_layout()
    plt.show()

    # Summary plot
    plt.figure()
    shap.plots.beeswarm(shap_values, max_display=15, show=False)
    plt.title(f"SHAP Summary - {nom_cible}")
    plt.tight_layout()
    plt.show()

    return shap_values

In [ ]:
# Analyse du débit
shap_debit = analyser_shap(
    model_debit,
    X_test_debit,
    "Débit"
)

# Analyse de l'occupation
shap_occupation = analyser_shap(
    model_occupation,
    X_test_occupation,
    "Occupation"
)